# SupplyMind AI — Random Forest

Model selection is performed on validation data only.

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
from supplymind.features.predictions.ml.evaluation import (
    choose_threshold,
    evaluate_probabilities,
    positive_class_probability,
)
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.reporting import (
    save_evaluation_plots,
    save_feature_importance,
    save_json,
)
from supplymind.features.predictions.ml.training import (
    build_random_forest,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=False,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_random_forest()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    ["f1", "recall"],
    ascending=False,
).head(10)

Selected threshold: 0.3300000000000001


,accuracy,precision,recall,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,false_positive_rate,false_negative_rate,threshold
13,0.577370,0.577324,0.999703,0.731950,0.739477,0.835119,8,9853,4,13458,0.999189,0.000297,0.33
11,0.577241,0.577230,0.999926,0.731934,0.739477,0.835119,2,9859,1,13461,0.999797,0.000074,0.31
0,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.20
1,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.21
2,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.22
3,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.23
4,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.24
5,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.25
6,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.26
7,0.577198,0.577198,1.000000,0.731929,0.739477,0.835119,0,9861,0,13462,1.000000,0.000000,0.27


In [7]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.5773699781331733,
 'precision': 0.5773240101239758,
 'recall': 0.9997028673302629,
 'f1': 0.7319500720637424,
 'roc_auc': 0.7394773912125235,
 'average_precision': 0.8351188650637533,
 'true_negative': 8,
 'false_positive': 9853,
 'false_negative': 4,
 'true_positive': 13458,
 'false_positive_rate': 0.9991887232532197,
 'false_negative_rate': 0.0002971326697370376,
 'threshold': 0.3300000000000001}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "random_forest"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)